# 🩺 Kidney Disease 4-Class Classification using Fine-Tuned VGG-16

This notebook demonstrates the end-to-end training pipeline for classifying Kidney CT Scan images into 4 distinct diagnostic categories:
1. **Cyst**
2. **Normal**
3. **Stone**
4. **Tumor**

### ⚙️ Pipeline Overview:
- Load pre-trained **VGG-16** weights (ImageNet).
- Unfreeze top layers for domain adaptation.
- Attach custom Dense head with BatchNormalization and Dropout.
- Train across epochs using Keras Callbacks (`ModelCheckpoint`, `EarlyStopping`, `ReduceLROnPlateau`, `CSVLogger`).
- Save trained weights directly to **`model/model.h5`**.

## 1. Import Dependencies & Load Hyperparameters

In [ ]:
import os
import json
import yaml
import numpy as np
import tensorflow as tf
from pathlib import Path

# Load hyperparameters
with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

IMAGE_SIZE = tuple(params['IMAGE_SIZE'])
BATCH_SIZE = params['BATCH_SIZE']
EPOCHS = params['EPOCHS']
CLASSES = params['CLASSES']
LEARNING_RATE = params['LEARNING_RATE']
FREEZE_TILL = params['FREEZE_TILL']
DROPOUT = params['DROPOUT']

os.makedirs('../model', exist_ok=True)
os.makedirs('../logs', exist_ok=True)
print(f"Loaded hyperparameters — Epochs: {EPOCHS}, Classes: {CLASSES}, LR: {LEARNING_RATE}")

## 2. Dataset Preparation & Data Augmentation

In [ ]:
dataset_dir = '../artifacts/data_ingestion/Kindey_Stone_Dataset'
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'val')
test_dir = os.path.join(dataset_dir, 'test')

# Validation Generator (Rescaling only)
val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)
val_gen = val_datagen.flow_from_directory(
    directory=val_dir,
    target_size=IMAGE_SIZE[:-1],
    batch_size=BATCH_SIZE,
    interpolation='bilinear',
    shuffle=False
)

# Training Generator (With Data Augmentation)
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=40,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    fill_mode='nearest'
)

train_gen = train_datagen.flow_from_directory(
    directory=train_dir,
    target_size=IMAGE_SIZE[:-1],
    batch_size=BATCH_SIZE,
    interpolation='bilinear',
    shuffle=True
)

# Save Class Index Mapping directly into model/ folder
class_indices = train_gen.class_indices
class_names = {v: k for k, v in class_indices.items()}
with open('../model/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=4)

print(f"✅ Class Mapping Saved to model/class_names.json: {class_names}")

## 3. Build & Fine-Tune VGG-16 Architecture

In [ ]:
# Load VGG16 Backbone with pre-trained ImageNet weights
base_model = tf.keras.applications.vgg16.VGG16(
    input_shape=IMAGE_SIZE,
    weights='imagenet',
    include_top=False
)

# Freeze early feature extraction layers
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze top 4 layers for domain fine-tuning
if FREEZE_TILL > 0:
    for layer in base_model.layers[-FREEZE_TILL:]:
        layer.trainable = True

# Custom Classification Head
x = tf.keras.layers.Flatten()(base_model.output)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(DROPOUT)(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(DROPOUT * 0.6)(x)
output = tf.keras.layers.Dense(units=CLASSES, activation='softmax')(x)

model = tf.keras.models.Model(inputs=base_model.input, outputs=output)

# Compile Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

## 4. Train Model & Save Directly to `model/model.h5`

In [ ]:
steps_per_epoch = train_gen.samples // BATCH_SIZE
val_steps = val_gen.samples // BATCH_SIZE

callbacks = [
    # 📌 Save best model directly into model/model.h5
    tf.keras.callbacks.ModelCheckpoint(
        filepath='../model/model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger('../logs/training_log.csv', separator=',', append=False)
]

print(f"🚀 Starting Model Training for {EPOCHS} Epochs...")
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    validation_data=val_gen,
    callbacks=callbacks
)

# Ensure model is stored in model/model.h5
model.save('../model/model.h5')
print("💾 Trained model saved to model/model.h5!")

## 5. Model Evaluation & Per-Class Metrics

In [ ]:
# Load test generator
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)
test_gen = test_datagen.flow_from_directory(
    directory=test_dir,
    target_size=IMAGE_SIZE[:-1],
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Evaluate on unseen test set
scores = model.evaluate(test_gen)
predictions = model.predict(test_gen)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_gen.classes

# Calculate Per-Class Precision, Recall, F1-Score
per_class_metrics = {}
confusion = np.zeros((len(class_names), len(class_names)), dtype=int)

for true, pred in zip(true_classes, predicted_classes):
    confusion[true][pred] += 1

for i in range(len(class_names)):
    name = class_names[i]
    tp = np.sum((predicted_classes == i) & (true_classes == i))
    fp = np.sum((predicted_classes == i) & (true_classes != i))
    fn = np.sum((predicted_classes != i) & (true_classes == i))

    prec = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
    rec = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    f1 = float(2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

    per_class_metrics[name] = {
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1_score": round(f1, 4),
        "support": int(np.sum(true_classes == i))
    }

score_data = {
    "loss": round(float(scores[0]), 4),
    "accuracy": round(float(scores[1]), 4),
    "per_class_metrics": per_class_metrics
}

with open('../logs/scores.json', 'w') as f:
    json.dump(score_data, f, indent=4)

print(f"🎉 Final Test Accuracy: {score_data['accuracy'] * 100:.2f}%")
print(json.dumps(per_class_metrics, indent=4))